In [1]:
from pathlib import Path
import os
import json

from pprint import pprint

from utils.generator import run_generator, load_json, save_json
from utils.assignment import assign_character_ids, assign_roles_to_characters, validate_relationships_artifact, validate_motives_artifact

In [ ]:
GEMINI_API_KEY = "" 
# os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [3]:
TEMPLATE_PATH = Path("./templates/")
PROMPTS_PATH = Path("./prompts/")
SCHEMAS_PATH = Path("./schemas/")
ARTIFACTS_PATH = Path("./artifacts/")

MODEL_NAME = "gemini-2.5-flash"

## Settings Generation

In [4]:
from prompts.setting_prompt import build_settings_prompt
from schemas.setting_schema import Settings

In [ ]:
settings = run_generator(
    template_path=TEMPLATE_PATH / "settings_base.json",
    prompt_builder=build_settings_prompt,
    schema_model=Settings,
    output_path=ARTIFACTS_PATH / "settings.json",
    api_key=GEMINI_API_KEY,
)

In [5]:
settings_artifact = load_json(ARTIFACTS_PATH / "settings.json")
pprint(settings_artifact)

{'character_settings': {'complexity': 'moderate', 'suspects': 5},
 'story_settings': {'intended_audience': 'adult',
                    'point_of_view': 'third person limited',
                    'reading_level': 'adult general',
                    'sub_genre': 'cozy mystery',
                    'writing_style': 'witty and charming'},
 'world_settings': {'location': 'a picturesque coastal town in Maine',
                    'realism_level': 'grounded',
                    'time_period': 'present day'}}


## Character Generation

In [ ]:
from prompts.character_prompt import build_character_prompt
from schemas.character_schema import CharacterArtifactLLM

In [ ]:
characters_result = run_generator(
    template_path=TEMPLATE_PATH /"character_base.json",
    prompt_builder=build_character_prompt,
    schema_model=CharacterArtifactLLM,
    output_path=ARTIFACTS_PATH / "characters.json",
    context_artifacts={
        "settings_artifact": settings_artifact
    },
    model_name=MODEL_NAME,
    temperature=0.9,
    api_key=GEMINI_API_KEY,
)

In [ ]:
characters = assign_character_ids(characters_result)
save_json(characters.model_dump(), ARTIFACTS_PATH / "characters.json")

In [6]:
characters_artifact = load_json(ARTIFACTS_PATH / "characters.json")
print("Num Chars:", len(characters_artifact["characters"]))
pprint(characters_artifact)

Num Chars: 7
{'characters': [{'background': {'notable_past_events': ['Restored and opened '
                                                        "The Gull's Perch Inn",
                                                        "Husband's sudden "
                                                        'passing five years '
                                                        'prior',
                                                        'Financial struggles '
                                                        "with the inn's "
                                                        'upkeep'],
                                'origin': 'Nearby city in Maine',
                                'personal_history_summary': 'Moved to the '
                                                            'picturesque '
                                                            'coastal town '
                                                            'twenty years ago '
                

## Role Assignment

In [7]:
from prompts.role_prompt import build_role_assignment_prompt
from schemas.role_schema import RoleAssignmentArtifact

In [ ]:
role_assignment_result = run_generator(
    template_path=TEMPLATE_PATH / "roles_base.json",
    prompt_builder=build_role_assignment_prompt,
    schema_model=RoleAssignmentArtifact,
    output_path=ARTIFACTS_PATH / "role_assignment.json",
    context_artifacts={
        "settings_artifact": settings_artifact,
        "characters_artifact": characters_artifact,
    },
    model_name=MODEL_NAME,
    temperature=0.8,
    api_key=GEMINI_API_KEY,
)

In [ ]:
updated_characters = assign_roles_to_characters(
    characters_artifact,
    role_assignment_result,
)

save_json(updated_characters.model_dump(), ARTIFACTS_PATH / "characters.json")

In [8]:
characters_artifact = load_json(ARTIFACTS_PATH / "characters.json")
print("Num Chars:", len(characters_artifact["characters"]))
pprint(characters_artifact)

Num Chars: 7
{'characters': [{'background': {'notable_past_events': ['Restored and opened '
                                                        "The Gull's Perch Inn",
                                                        "Husband's sudden "
                                                        'passing five years '
                                                        'prior',
                                                        'Financial struggles '
                                                        "with the inn's "
                                                        'upkeep'],
                                'origin': 'Nearby city in Maine',
                                'personal_history_summary': 'Moved to the '
                                                            'picturesque '
                                                            'coastal town '
                                                            'twenty years ago '
                

## Relationship Generation

In [9]:
from prompts.relationship_prompt import build_relationship_prompt
from schemas.relationship_schema import RelationshipArtifact

In [ ]:
relationships_result = run_generator(
    template_path=TEMPLATE_PATH / "relationships_base.json",
    prompt_builder=build_relationship_prompt,
    schema_model=RelationshipArtifact,
    output_path=ARTIFACTS_PATH / "relationships.json",
    context_artifacts={
        "settings_artifact": settings_artifact,
        "characters_artifact": characters_artifact,
    },
    model_name=MODEL_NAME,
    temperature=0.8,
    api_key=GEMINI_API_KEY,
)

In [ ]:
relationships_result = validate_relationships_artifact(
    relationships_result,
    characters_artifact,
)

save_json(relationships_result.model_dump(), ARTIFACTS_PATH / "relationships.json")

In [10]:
relationships_artifact = load_json(ARTIFACTS_PATH / "relationships.json")
print("Num relations:", len(relationships_artifact["relationships"]))
pprint(relationships_artifact)

Num relations: 6
{'relationships': [{'character_a_id': 'char_01',
                    'character_a_private_view': 'Views Beatrice as a polite, '
                                                'inquisitive new resident, '
                                                'perhaps a bit nosy but '
                                                'generally harmless. '
                                                'Appreciates her patronage and '
                                                'interest in the local '
                                                'community.',
                    'character_b_id': 'char_07',
                    'character_b_private_view': 'Views Eleanor as a gracious, '
                                                'resilient woman, a keeper of '
                                                'local history, and a '
                                                'potential character study for '
                                                'her no

## Motives Generation

In [11]:
from prompts.motive_prompt import build_motive_prompt
from schemas.motive_schema import MotiveArtifact

In [12]:
motives_result = run_generator(
    template_path=TEMPLATE_PATH / "motives_base.json",
    prompt_builder=build_motive_prompt,
    schema_model=MotiveArtifact,
    output_path=ARTIFACTS_PATH / "motives.json",
    context_artifacts={
        "settings_artifact": settings_artifact,
        "characters_artifact": characters_artifact,
        "relationships_artifact": relationships_artifact
    },
    model_name=MODEL_NAME,
    temperature=0.9,
    api_key=GEMINI_API_KEY,
)

In [13]:
motives_result = validate_motives_artifact(
    motives_result,
    characters_artifact
)
save_json(motives_result.model_dump(), ARTIFACTS_PATH / "motives.json")

In [15]:
motives_artifact = load_json(ARTIFACTS_PATH / "motives.json")
print("Num relations:", len(motives_artifact["motives"]))
pprint(motives_artifact)

Num relations: 6
{'motives': [{'character_id': 'char_07',
              'character_role': 'detective',
              'is_red_herring': 'false',
              'motive_strength': 'moderate',
              'motive_summary': "As a mystery writer struggling with writer's "
                                'block, Beatrice is driven by her natural '
                                'curiosity and a personal sense of duty to '
                                'uncover the truth and bring justice for '
                                'Eleanor.',
              'motive_type': 'positive',
              'target_character_id': 'char_01'},
             {'character_id': 'char_02',
              'character_role': 'suspect',
              'is_red_herring': 'true',
              'motive_strength': 'moderate',
              'motive_summary': "Elias resented Eleanor and The Gull's Perch "
                                "Inn for symbolizing the town's shift towards "
                                'tourism 